In [103]:
import numpy as np
import pandas as pd
from file_paths import *

In [104]:
# gold standard
df_gs = pd.read_excel(join(INPUT_DIRECTORY_UC2_GS, "gs_uc2_subprocess_level.xlsx")) 
# output by ranking algorithm
df_alg = pd.read_excel(join(INPUT_DIRECTORY_UC2_ALGO, "uc2_subprocess_level_algo_output_bi_ce.xlsx")) 

In [105]:
df_gs.head()

,query,rel_text
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","The reporting entity must, as soon as practicable, take reasonable measures to:\n\n(1) obtain and verify additional KYC information; or\n\n(2) update and verify existing KYC information;\n\nso that the reporting entity is reasonably satisfied that the customer, beneficial owner or person purporting to act on behalf of the customer is the person that the customer, beneficial owner or person purporting to act on behalf of the customer claims to be.\n\nNote: A reporting entity is not required to take any measures that would contravene the tipping off offence in section 123 of the Act."
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include appropriate risk‑based systems and controls that are designed to enable the reporting entity to be reasonably satisfied, where a customer is an individual, that the customer is the individual that he or she claims to be."
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules.\n\n"
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect, at a minimum, the following KYC information about an individual (other than an individual who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader):\n\n(1) the customer’s full name;\n\n(2) the customer’s date of birth; and\n\n(3) the customer’s residential address."
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect at a minimum, the following KYC information about a customer who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader:\n\n(1) the customer’s full name;\n\n(2) the customer’s date of birth;\n\n(3) the full business name (if any) under which the customer carries on his or her business;\n\n(4) the full address of the customer’s principal place of business (if any) or the customer’s residential address; and\n\n(5) any ABN issued to the customer."


In [106]:
df_gs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   query     60 non-null     object
 1   rel_text  60 non-null     object
dtypes: object(2)
memory usage: 1.1+ KB


In [107]:
def clean_text(text):  
    '''cleans texts'''
    cleaned_text = text.replace("or\n\n\n", " ")
    cleaned_text = cleaned_text.replace("or\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n", " ")
    cleaned_text = cleaned_text.replace("\n \n", " ")
    cleaned_text = cleaned_text.replace("\n", " ")
    return cleaned_text 


In [108]:
df_gs['query_cleaned'] = df_gs.apply(lambda row : clean_text(row['query']), axis = 1)
df_gs['rel_text_cleaned'] = df_gs.apply(lambda row : clean_text(row['rel_text']), axis = 1)
df_gs = df_gs.drop(['query', 'rel_text'], axis=1)
df_gs = df_gs.rename(columns={'query_cleaned': 'query', 'rel_text_cleaned': 'rel_text'})
df_gs.head()

,query,rel_text
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","The reporting entity must, as soon as practicable, take reasonable measures to: (1) obtain and verify additional KYC information; (2) update and verify existing KYC information; so that the reporting entity is reasonably satisfied that the customer, beneficial owner or person purporting to act on behalf of the customer is the person that the customer, beneficial owner or person purporting to act on behalf of the customer claims to be. Note: A reporting entity is not required to take any measures that would contravene the tipping off offence in section 123 of the Act."
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include appropriate risk‑based systems and controls that are designed to enable the reporting entity to be reasonably satisfied, where a customer is an individual, that the customer is the individual that he or she claims to be."
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules."
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect, at a minimum, the following KYC information about an individual (other than an individual who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader): (1) the customer’s full name; (2) the customer’s date of birth; (3) the customer’s residential address."
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect at a minimum, the following KYC information about a customer who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader: (1) the customer’s full name; (2) the customer’s date of birth; (3) the full business name (if any) under which the customer carries on his or her business; (4) the full address of the customer’s principal place of business (if any) or the customer’s residential address; (5) any ABN issued to the customer."


In [109]:
df_alg.head()

,query,rel_text,score
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",We require our Investigators to: a. record the requests they make to individuals for written authorisation to access the individual’s personal information that is held by other parties; and b. to provide those records to us at the end of their investigation.,-3.187567
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","If a reporting entity is unable to establish the identity of a customer using the applicable customer identification requirements specified in Chapter 4 of the AML/CTF Rules because the customer does not possess, and is unable to obtain, the necessary information or evidence of identity, then it may use alternative identity proofing processes, in accordance with its risk-based systems and controls, to do so.",-3.708699
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",An AML/CTF program must require that the verification of information collected about a customer be based on: (1) reliable and independent documentation; (2) reliable and independent electronic data; or (3) a combination of (1) and (2) above.,-3.715315
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",An Australian Privacy Principle entity must collect personal information only by lawful and fair means.,-4.417554
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","If a reporting entity is unable to establish the identity of a customer in accordance with paragraph 4.15.1 or 4.15.1A, then it may accept a self-attestation from the customer certifying that the information provided in relation to their identity is true and correct.",-4.475100


In [110]:
df_alg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   query     140 non-null    object 
 1   rel_text  140 non-null    object 
 2   score     140 non-null    float64
dtypes: float64(1), object(2)
memory usage: 3.4+ KB


In [111]:
# add column rank to df_alg
df_alg["rank"] = df_alg.groupby("query")["score"].rank(ascending=False)
df_alg.head()

,query,rel_text,score,rank
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",We require our Investigators to: a. record the requests they make to individuals for written authorisation to access the individual’s personal information that is held by other parties; and b. to provide those records to us at the end of their investigation.,-3.187567,1.0
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","If a reporting entity is unable to establish the identity of a customer using the applicable customer identification requirements specified in Chapter 4 of the AML/CTF Rules because the customer does not possess, and is unable to obtain, the necessary information or evidence of identity, then it may use alternative identity proofing processes, in accordance with its risk-based systems and controls, to do so.",-3.708699,2.0
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",An AML/CTF program must require that the verification of information collected about a customer be based on: (1) reliable and independent documentation; (2) reliable and independent electronic data; or (3) a combination of (1) and (2) above.,-3.715315,3.0
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.",An Australian Privacy Principle entity must collect personal information only by lawful and fair means.,-4.417554,4.0
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","If a reporting entity is unable to establish the identity of a customer in accordance with paragraph 4.15.1 or 4.15.1A, then it may accept a self-attestation from the customer certifying that the information provided in relation to their identity is true and correct.",-4.475100,5.0


In [112]:
# merge dataframes
df_gs_enhanced = pd.merge(df_gs, df_alg,  how='left', left_on=['query','rel_text'], right_on = ['query','rel_text'])
df_gs_enhanced.tail()

,query,rel_text,score,rank
55,"To onboard a new customer, their account is created in the bank's system, relevant agreements and disclosures are signed, and the features and benefits of the customer's specific account type, such as interest rates and fees, are explained to them.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules.",NaN,NaN
56,"To welcome a new customer, an overview of the bank's products and services that may be of interest to the customer based on their profile is provided, while assistance is offered to the customer in navigating their account, including setting up online banking, setting up alerts, and addressing any questions or concerns they may have.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules.",NaN,NaN
57,"To welcome a new customer, an overview of the bank's products and services that may be of interest to the customer based on their profile is provided, while assistance is offered to the customer in navigating their account, including setting up online banking, setting up alerts, and addressing any questions or concerns they may have.","If an Australian Privacy Principle entity holds personal information about an individual, the entity must, on request by the individual, give the individual access to the information.",NaN,NaN
58,"To welcome a new customer, an overview of the bank's products and services that may be of interest to the customer based on their profile is provided, while assistance is offered to the customer in navigating their account, including setting up online banking, setting up alerts, and addressing any questions or concerns they may have.","If: a) the Australian Privacy Principle entity is an agency; and b) the entity is required or authorised to refuse to give the individual access to the personal information by or under: - the Freedom of Information Act; or - any other Act of the Commonwealth, or a Norfolk Island enactment, that provides for access by persons to documents; then, despite subclause 12.1, the entity is not required to give access to the extent that the entity is required or authorised to refuse to give access.",NaN,NaN
59,"To welcome a new customer, an overview of the bank's products and services that may be of interest to the customer based on their profile is provided, while assistance is offered to the customer in navigating their account, including setting up online banking, setting up alerts, and addressing any questions or concerns they may have.","The Australian Privacy Principle entity must: a) respond to the request for access to the personal information: - if the entity is an agency — within 30 days after the request is made; or - if the entity is an organisation — within a reasonable period after the request is made; and b) give access to the information in the manner requested by the individual, if it is reasonable and practicable to do so.",NaN,NaN


In [113]:
# average precision (AP)
df_gs_enhanced["AP"] = 1/df_gs_enhanced["rank"]
df_gs_enhanced.head()

,query,rel_text,score,rank,AP
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","The reporting entity must, as soon as practicable, take reasonable measures to: (1) obtain and verify additional KYC information; (2) update and verify existing KYC information; so that the reporting entity is reasonably satisfied that the customer, beneficial owner or person purporting to act on behalf of the customer is the person that the customer, beneficial owner or person purporting to act on behalf of the customer claims to be. Note: A reporting entity is not required to take any measures that would contravene the tipping off offence in section 123 of the Act.",NaN,NaN,NaN
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include appropriate risk‑based systems and controls that are designed to enable the reporting entity to be reasonably satisfied, where a customer is an individual, that the customer is the individual that he or she claims to be.",NaN,NaN,NaN
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules.",NaN,NaN,NaN
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect, at a minimum, the following KYC information about an individual (other than an individual who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader): (1) the customer’s full name; (2) the customer’s date of birth; (3) the customer’s residential address.",NaN,NaN,NaN
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect at a minimum, the following KYC information about a customer who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader: (1) the customer’s full name; (2) the customer’s date of birth; (3) the full business name (if any) under which the customer carries on his or her business; (4) the full address of the customer’s principal place of business (if any) or the customer’s residential address; (5) any ABN issued to the customer.",NaN,NaN,NaN


In [114]:
df_gs_enhanced = df_gs_enhanced.fillna(0)
df_gs_enhanced.head()

,query,rel_text,score,rank,AP
0,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","The reporting entity must, as soon as practicable, take reasonable measures to: (1) obtain and verify additional KYC information; (2) update and verify existing KYC information; so that the reporting entity is reasonably satisfied that the customer, beneficial owner or person purporting to act on behalf of the customer is the person that the customer, beneficial owner or person purporting to act on behalf of the customer claims to be. Note: A reporting entity is not required to take any measures that would contravene the tipping off offence in section 123 of the Act.",0.0,0.0,0.0
1,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include appropriate risk‑based systems and controls that are designed to enable the reporting entity to be reasonably satisfied, where a customer is an individual, that the customer is the individual that he or she claims to be.",0.0,0.0,0.0
2,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","In so far as a reporting entity has any customer who is an individual, an AML/CTF program must comply with the requirements specified in Part 4.2 of these Rules.",0.0,0.0,0.0
3,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect, at a minimum, the following KYC information about an individual (other than an individual who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader): (1) the customer’s full name; (2) the customer’s date of birth; (3) the customer’s residential address.",0.0,0.0,0.0
4,"To create a new account, the required personal information and documents are gathered from the customer and entered into the bank's account opening system, and a unique account number is generated to identify the account in the bank's systems.","An AML/CTF program must include a procedure for the reporting entity to collect at a minimum, the following KYC information about a customer who notifies the reporting entity that he or she is a customer of the reporting entity in his or her capacity as a sole trader: (1) the customer’s full name; (2) the customer’s date of birth; (3) the full business name (if any) under which the customer carries on his or her business; (4) the full address of the customer’s principal place of business (if any) or the customer’s residential address; (5) any ABN issued to the customer.",0.0,0.0,0.0


In [115]:
# mean average precision (MAP)
df_gs_enhanced["AP"].mean()

0.041066091774296105

In [116]:
# count per query: average number gs rank > 0 (=TP) 
df_test = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x>0).sum()).reset_index(name='count')
df_test.loc[:, 'count'].mean()

2.0

In [117]:
# count per query: average number gs rank == 0 (=FN) 
df_test = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x==0).sum()).reset_index(name='count')
df_test.loc[:, 'count'].mean()

6.571428571428571

In [118]:
df_gs_enhanced.to_excel(join(RESULT_DIRECTORY, "results_subprocess_uc2_bi_ce.xlsx"))

In [119]:
pd.options.display.max_colwidth = 1000